# 01. 시계열 데이터의 이해와 실습 토대

> **Day 01 — 제조 시계열 AI (2/5)**
> 제조 센서 시계열을 다루는 기초 체력을 만든다. 여기서 만든 함수는
> NB02~NB04에서 계속 재사용한다.
>
> **이 노트북의 역할**: 모든 실습의 토대. 특히 슬라이딩 윈도우와 시간 기반 분할은
> 이후 모든 모델링의 전제가 된다.

---

## 📖 이 노트북의 스토리라인

> **"시계열은 행을 섞으면 죽는 데이터입니다. 그래서 다루는 법부터 다릅니다."**

```
왜 다른가              어떻게 다루는가                        무엇을 남기는가
순서·자기상관·비정상성  →  리샘플링 → 결측 → 홀드값 → ACF  →  슬라이딩 윈도우 + 시간 분할
```

| | |
|---|---|
| **쓰는 데이터** | CNC 스핀들 진동·부하, 생산라인 전력 (모두 합성 — 결측·홀드값·이상을 **의도적으로 심어야** 하므로) |
| **이 노트북의 역할** | 이후 전 실습의 **토대**. 여기서 만든 함수를 NB02~NB04에서 계속 재사용한다 |
| **앞에서 이어받는 것** | NB00에서 발견한 결측 블록·홀드값 — 여기서 잡는 법을 배운다 |
| **다음으로 넘기는 것** | `make_windows()` → NB02의 **Patch 개념**으로 직결. 시간 분할 원칙 → NB02~04 전체 |

> 화려한 모델은 없습니다. 하지만 **여기서 만든 습관이 오후의 성패를 가릅니다.**
> 특히 §9의 데이터 누수 실습은 오늘 하루 중 가장 실무적인 장면입니다.

---

> **실습 안내**
> `"""따라하기"""` 가 적힌 셀은 강사와 함께 직접 실행합니다. 주석을 보고 코드를 채워 주세요.
> `"""직접구현"""` 이 적힌 셀은 여러분이 직접 채워 봅니다. 정답은 노트북 맨 아래에 있습니다.
> 나머지 셀은 실행 결과를 확인하며 따라오시면 됩니다.
> 막히는 부분은 손을 들어 주세요.

## 목차
1. 시계열은 왜 다른가 — 행을 섞으면 죽는 데이터
2. 왜 중요한가 — 사후보전과 예지보전
3. 제조 센서 데이터의 얼굴
4. 시간 인덱싱과 리샘플링
5. 결측 처리 — 채울 것인가, 제외할 것인가
6. 홀드값과 스파이크 — 제조 데이터 특유의 함정
7. 자기상관(ACF) 훑어보기
8. 슬라이딩 윈도우 데이터셋 생성 ★
9. 시간 기반 분할 ★ — 누수를 눈으로 확인
10. 스케일링 — train에서만 fit

In [ ]:
# 공통 준비 — Colab / 로컬 양쪽에서 동작
import os, random, warnings
import numpy as np
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} | device: {DEVICE}")
except ImportError:
    DEVICE = "cpu"
    print("PyTorch 미설치 — 다음 셀에서 설치합니다.")

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print(f"Colab 환경: {IN_COLAB}")

In [ ]:
# 패키지 설치 (Colab 기본 탑재 외 추가분만)
!pip install -qU statsmodels

In [ ]:
# ══ 실습 자료 위치 — 강사가 배포한 주소로 이 한 줄만 맞추면 됩니다 ══════════
DATA_REPO = "https://raw.githubusercontent.com/leejiyoon52/ai-course/main/day1"

# Google Drive로 배포받았다면, 위 줄 대신 아래 두 줄의 주석을 푸십시오.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_REPO = "file:///content/drive/MyDrive/ai-course/day1"
# ═══════════════════════════════════════════════════════════════════════

import os, shutil, urllib.request
os.environ["MFG_DATA_BASE"] = DATA_REPO          # loaders.py가 이 값을 읽습니다

for f in ["mfg_datagen.py", "loaders.py"]:
    if os.path.exists(f):
        continue
    src = f"{DATA_REPO}/modules/{f}"
    try:
        if src.startswith("file://"):
            shutil.copy(src[len("file://"):], f)
        else:
            urllib.request.urlretrieve(src, f)
        print(f"다운로드 완료: {f}")
    except Exception:
        print(f"⚠️ {f} 를 받지 못했습니다 — 왼쪽 파일 탭에 직접 업로드해 주세요")

import mfg_datagen
import loaders
print(f"실습 모듈 준비 완료 — 자료 위치: {DATA_REPO}")

---
## 1. 시계열은 왜 다른가 — 행을 섞으면 죽는 데이터

일반 정형 데이터는 행의 순서를 바꿔도 정보가 보존됩니다. 시계열은 다릅니다.

- **순서**: 값의 의미가 "언제"에 묶여 있습니다.
- **자기상관**: 지금 값은 조금 전 값과 닮아 있습니다.
- **비정상성**: 평균·분산 자체가 시간에 따라 변합니다 (마모, 계절, 교대조).

말로 하면 심심하니, 실제로 행을 섞어서 무엇이 죽는지 보겠습니다.
데이터는 **CNC 스핀들**(공작기계의 회전축) 진동·부하 센서입니다.

In [ ]:
# CNC 스핀들 데이터 생성 — 48시간, 10초 샘플링
"""따라하기"""


In [ ]:
# 행을 무작위로 섞으면 무엇이 사라지는지 비교한다
"""따라하기"""


In [ ]:
# 자기상관을 숫자로 — "지금 값은 조금 전 값과 닮았다"
for lag in [1, 6, 60]:
    r = cnc["vib_rms"].autocorr(lag=lag)
    print(f"lag {lag:3d} ({lag * 10:4d}초 전)과의 상관: {r:+.3f}")
print("섞은 뒤에는:", round(shuffled.autocorr(lag=1), 3), "— 이웃 관계가 끊어졌습니다")

---
## 2. 왜 중요한가 — 사후보전과 예지보전

> 시계열은 **설비의 심전도**입니다. 파형 자체가 상태를 말하고, **고장은 신호에 먼저 나타납니다.**

| | 사후보전 (Run-to-Failure) | 예지보전 (PdM) |
|---|---|---|
| 대응 시점 | 멈춘 뒤 | 신호가 이상해진 시점 |
| 비용 | 비계획 정지 + 연쇄 지연 + 긴급 수리 | 계획 정비 (생산 계획에 흡수) |
| 필요한 것 | 없음 | **신호를 읽는 능력** ← 오늘 배우는 것 |

아래 그래프에서 공구 마모가 진동 신호에 어떻게 미리 나타나는지 확인합니다.

In [ ]:
# 공구 마모(tool_age)와 진동(vib_rms)의 동행 — 고장은 신호에 먼저 나타난다
fig, ax1 = plt.subplots(figsize=(12, 3.5))
ax1.plot(cnc.index, cnc["vib_rms"], lw=0.4, label="vib_rms")
ax1.set_ylabel("vib_rms (mm/s)")
ax2 = ax1.twinx()
ax2.plot(cnc.index, cnc["tool_age_min"], color="tab:orange", lw=1.2, label="tool_age")
ax2.set_ylabel("tool age (min)")
ax1.set_title("Vibration grows with tool wear, resets on tool change")
plt.tight_layout()
plt.show()
print("공구 교체(주황 톱니의 리셋) 직전마다 진동 진폭이 커져 있습니다 — 이 패턴이 예지보전의 근거입니다.")

In [ ]:
# 봉투 뒷면 계산 — 비계획 정지 1회와 계획 정비 1회의 비용 차이
HOURLY_LOSS = 300        # 라인 정지 시간당 기회손실 (만원, 예시)
unplanned = 6 * HOURLY_LOSS + 500 + 300   # 돌발 정지 6시간 + 긴급수리 + 불량 재작업
planned = 2 * HOURLY_LOSS + 200           # 계획 정비 2시간(야간 배치) + 정기 부품비

print(f"비계획 정지 1회: 약 {unplanned:,}만원")
print(f"계획 정비 1회  : 약 {planned:,}만원")
print(f"차이           : {unplanned - planned:,}만원 → 신호를 읽는 능력의 값어치입니다")
print("(숫자는 예시입니다 — 실제로는 공정·업종별 비용 구조로 다시 계산해야 합니다)")

---
## 3. 제조 센서 데이터의 얼굴

제조 시계열의 전형적인 리듬 세 가지 — **사이클, 교대조, 요일** — 을
생산라인 전력 데이터에서 확인합니다.

In [ ]:
# 생산라인 전력 데이터 생성 — 4주, 10분 샘플링
"""따라하기"""


In [ ]:
# 시간대·요일 프로파일 — 교대조와 요일 효과를 평균으로 요약
prof_h = energy.groupby(energy.index.hour)["power_kw"].mean()
prof_d = energy.groupby(energy.index.dayofweek)["power_kw"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
axes[0].plot(prof_h.index, prof_h.values, marker="o")
axes[0].set_title("Mean power by hour (shift pattern)")
axes[0].set_xlabel("hour")
axes[1].bar(range(7), prof_d.values)
axes[1].set_xticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
axes[1].set_title("Mean power by weekday")
plt.tight_layout()
plt.show()
print("08~20시 주간조 부하, 주말 감산 — 이 리듬을 모르면 주말 새벽 가동이 '정상'으로 보입니다.")

In [ ]:
# 1주만 확대 — 계획 정지(PM)와 이상 구간은 모양이 다르다
"""따라하기"""


---
## 4. 시간 인덱싱과 리샘플링

> **샘플링 주기**는 심전도를 몇 초에 한 번 찍는가와 같습니다 — 너무 성기면 부정맥을 놓칩니다.

`DatetimeIndex` 가 있으면 시간이 곧 조회 키가 됩니다.
그리고 서로 다른 주기의 센서를 정렬하려면 **리샘플링**이 필요합니다.
이때 집계 함수 선택(mean / max / last)이 곧 **어떤 정보를 남길 것인가**의 선택입니다.

In [ ]:
# 문자열 타임스탬프를 DatetimeIndex로 — 수집 로그를 시계열로 승격
"""따라하기"""


In [ ]:
# 시간으로 바로 썰어 보기 — 특정 날짜·시간대 조회
"""따라하기"""


In [ ]:
# 리샘플링 — 10초 → 10분. mean과 max는 남기는 정보가 다르다
"""따라하기"""


In [ ]:
# 서로 다른 주기의 센서 정렬 — 집계 함수를 컬럼별로 다르게 선택한다
aligned = cnc.resample("10min").agg({
    "vib_rms": "max",        # 이상 징후 보존이 목적 → max
    "spindle_load": "mean",  # 평균 부하 수준이 목적 → mean
    "tool_age_min": "last",  # 상태값(누적) → 구간 마지막 값
})
print(aligned.head().round(3).to_string())
print(f"\n10초 데이터 {len(cnc):,}행 → 10분 정렬 데이터 {len(aligned):,}행")

---
## 5. 결측 처리 — 채울 것인가, 제외할 것인가

제조 데이터의 결측은 한 점씩 흩어지지 않습니다. 통신 두절·수집기 재시작 때문에
**연속 블록**으로 옵니다. 블록형 결측 앞에서 보간(interpolation)은 위험해집니다.

In [ ]:
# 결측 블록의 위치와 길이를 찾는다 — run-length 계산
"""따라하기"""


In [ ]:
# 보간 방법 비교 — 긴 블록에서 보간은 '그럴듯한 거짓말'이 된다
"""따라하기"""


**판단 기준 — 채우지 말고 제외하는 선택**

- 짧은 결측(수 샘플): 보간해도 왜곡이 작습니다.
- **긴 블록**: 보간값은 학습 데이터를 오염시킵니다. **구간째 제외**가 안전합니다.
- 특히 이상탐지에서는 보간 구간이 "지나치게 매끈한 정상"으로 학습되어 탐지력을 깎아 먹습니다.

> **현장 노트**
> "불량 라벨"은 검사 시점 기준입니다. 실제 이상 발생 시점과는 시차가 있습니다.
> 검사가 2시간마다 돈다면, 라벨이 붙은 시점의 신호가 아니라 그 앞 구간 어딘가에
> 원인이 있습니다. 라벨을 믿고 시점을 그대로 쓰면 모델은 엉뚱한 구간을 학습합니다.
> 라벨 시차 보정은 화려한 모델링보다 성능을 더 크게 올려 주는 일이 많습니다.

In [ ]:
# 구간 제외 — 결측 블록 주변까지 유효 마스크로 걸러낸다
valid = cnc["vib_rms"].notna()
for start, length in blocks:
    loc = cnc.index.get_loc(start)
    valid.iloc[max(0, loc - 3): loc + length + 3] = False   # 블록 앞뒤 3샘플 여유

print(f"전체 {len(cnc):,}행 중 유효 {valid.sum():,}행 사용 ({100 * valid.mean():.1f}%)")
print("이 마스크는 8절 슬라이딩 윈도우에서 '결측 걸친 윈도우 버리기'로 다시 씁니다.")

---
## 6. 홀드값과 스파이크 — 제조 데이터 특유의 함정

> **홀드값(센서 고착)** 은 *멈춘 시계*입니다 — 하루 두 번은 맞지만 아무것도 알려주지 않습니다.

NaN은 눈에 보이지만, 홀드값은 **정상값처럼 생긴 죽은 데이터**라 더 위험합니다.
물리 범위(Range) 검증과 함께, 데이터가 "살아 있는지"를 확인하는 기본 검진 두 가지를 만듭니다.

In [ ]:
# 홀드값 탐지 함수 — 동일값이 k개 이상 반복되는 구간을 찾는다
"""따라하기"""


In [ ]:
# 물리 범위 검증과 스파이크 검진 — 센서 사양이 곧 1차 필터다
"""따라하기"""


In [ ]:
# 홀드 구간을 직접 확대해서 본다 — "일자로 죽은 선"
h_idx = cnc.index[hold_mask]
lo = cnc.index.get_loc(h_idx[0]) - 60
seg = cnc["spindle_load"].iloc[lo:lo + len(h_idx) + 120]

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(seg.index, seg.values, lw=0.8)
ax.axvspan(h_idx[0], h_idx[-1], color="red", alpha=0.15, label="hold (stuck)")
ax.set_title("Stuck sensor — flat line inside living signal")
ax.legend()
plt.tight_layout()
plt.show()
print("주변은 살아 숨쉬는데 이 구간만 죽어 있습니다 — 값은 '정상 범위'라 범위 검증으로는 못 잡습니다.")

---
## 7. 자기상관(ACF) 훑어보기

자기상관함수(ACF, Autocorrelation Function)는 "지금 값이 lag 시점 전 값과
얼마나 닮았는가"를 재는 도구입니다. **주기가 있으면 그 주기에서 산이 다시 솟습니다.**

여기서는 사이클 주기를 확인하는 용도로만 사용합니다. (시계열 분해·정상성 검정은 이 과정 범위 밖입니다)

In [ ]:
# ACF로 가공 사이클 주기 확인 — 산이 다시 솟는 lag를 읽는다
"""따라하기"""


---
## 8. 슬라이딩 윈도우 데이터셋 생성 ★

> 슬라이딩 윈도우는 *긴 영상을 일정 길이 클립으로 잘라 보는 것*입니다.

모델은 무한히 긴 시계열을 통째로 먹지 못합니다.
`input_len` 길이의 과거를 보고 `horizon` 만큼의 미래를 맞히는 (X, y) 쌍으로 썰어야 합니다.
**이 함수는 NB02의 Patch 개념으로 그대로 이어집니다** — Patch는 윈도우 안을 또 한 번 써는 것입니다.

In [ ]:
# 슬라이딩 윈도우 데이터셋 생성 — 오늘 하루 내내 재사용할 함수
"""따라하기"""


In [ ]:
# 5절의 유효 마스크 재사용 — 결측·홀드에 걸친 윈도우를 버린다
"""따라하기"""


In [ ]:
# 윈도우가 실제로 어떻게 겹치며 미끄러지는지 눈으로 확인
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(temp[:200], lw=0.8, color="gray", label="series")
for k, s0 in enumerate([0, 40, 80]):
    ax.plot(range(s0, s0 + 48), temp[s0:s0 + 48], lw=2, label=f"window {k}")
    ax.plot(range(s0 + 48, s0 + 54), temp[s0 + 48:s0 + 54], "r.", ms=6)
ax.set_title("Sliding windows (bold) and their targets (red dots)")
ax.legend()
plt.tight_layout()
plt.show()
print("한 칸씩 미끄러지며 (X, y) 쌍이 만들어집니다. 인접 윈도우는 대부분 겹칩니다 — 9절의 복선입니다.")

---
## 9. 시간 기반 분할 ★ — 누수를 눈으로 확인

> **데이터 누수(leakage)** 는 *시험 문제를 미리 보고 푼 모의고사 점수*입니다.

시계열에서 랜덤 셔플 분할은 금지입니다. 인접 윈도우가 겹치므로,
셔플하면 **검증셋의 답이 사실상 학습셋에 들어가 있게** 됩니다.

실험 무대는 예지보전에서 실제로 쓰는 **누적 손상 지표**(진동 에너지의 누적)입니다.
설비의 손상은 되돌아가지 않으므로 이 지표는 단조 증가합니다 —
즉 **미래는 항상 과거가 가 보지 않은 영역**입니다. 배포 상황과 정확히 같습니다.

In [ ]:
# 누적 손상 지표 만들기 — 단조 증가하는 열화 궤적
"""따라하기"""


In [ ]:
# [틀린 코드] 랜덤 셔플 분할 — 점수가 좋아 '보인다'
"""따라하기"""


In [ ]:
# [옳은 코드] 시간 순 분할 + gap — 진짜 실력이 드러난다
"""따라하기"""


In [ ]:
# 무슨 일이 벌어졌는지 그림으로 — 시간 분할 테스트 구간의 예측 vs 실제
pred = model.predict(Xd[tr_end + gap:])
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.plot(yd[tr_end + gap:, 0], label="actual", lw=1.5)
ax.plot(pred[:, 0], label="1-NN prediction", lw=1.2)
ax.set_title("Temporal test: model has never seen these damage levels")
ax.legend()
plt.tight_layout()
plt.show()
print("과거에서 답을 '찾아 쓰던' 모델은, 과거에 없는 미래 앞에서 멈춰 섭니다.")
print("셔플 평가는 이 장면을 영원히 보여주지 않습니다 — 그래서 금지입니다.")

**정리 — 시간 기반 분할의 원칙**

- train(과거) → valid(중간) → test(최근) 순서를 지킵니다.
- 경계에는 `input_len + horizon` 만큼 **gap** 을 둡니다 — 겹친 윈도우가 경계를 넘어 답을 흘리지 못하게.
- 배포 후 모델이 마주하는 상황(과거로 미래 예측)과 평가 조건을 일치시키는 일입니다.

> **현장 노트**
> PoC에서 잘 나오던 모델이 양산 적용에서 무너지는 전형적인 원인 1위가 바로 이 누수입니다.
> 검증 점수 계산기가 아니라 "이 점수는 배포 상황을 흉내 낸 점수인가"를 묻는 습관이
> 모델 구조 선택보다 먼저입니다. 저는 남의 PoC 결과를 볼 때 분할 코드부터 요청합니다.

---
## 10. 스케일링 — train에서만 fit

스케일러도 데이터에서 통계량(평균·표준편차)을 "학습"합니다.
전체 데이터로 `fit` 하면 미래의 통계가 과거로 새어 들어갑니다 — 또 하나의 누수입니다.

In [ ]:
# 스케일러는 train에서만 fit, valid/test에는 transform만
"""따라하기"""


---
## 정리 — 오늘 만든 도구 상자

| 도구 | 용도 | 다시 쓰는 곳 |
|---|---|---|
| `find_na_blocks()` | 블록형 결측 위치·길이 | 데이터 검진 루틴 |
| `detect_hold()` | 센서 고착 탐지 | 데이터 검진 루틴 |
| `make_windows()` | (X, y) 윈도우 생성 | **NB02 LSTM·PatchTST 입력** |
| 시간 분할 + gap | 누수 없는 평가 | **NB02~04 전 실습** |
| train-only 스케일링 | 누수 없는 정규화 | **NB02~04 전 실습** |

---
## Self-check

### Q1. 시계열 데이터를 학습/검증 분할할 때 랜덤 셔플을 쓰면 안 되는 이유는?

<details>
<summary>정답 보기</summary>

- 미래 시점의 정보가 학습셋에 섞여 들어가 **데이터 누수(leakage)** 가 발생합니다.
- 인접 윈도우가 서로 겹치기 때문에, 셔플하면 검증 문제의 답이 학습셋에 사실상 존재합니다.
- 검증 성능은 좋게 나오지만 실제 배포 시 성능이 급락합니다.
- 반드시 **시간 순서 기준 분할**(train: 과거 → valid: 중간 → test: 최근)과 경계 gap을 사용합니다.

</details>

---

### Q2. 긴 블록형 결측을 선형 보간으로 채우면 어떤 문제가 생깁니까?

<details>
<summary>정답 보기</summary>

- 보간선은 실제 신호의 리듬·분산을 재현하지 못해 "지나치게 매끈한 가짜 정상"을 만듭니다.
- 이상탐지 모델이 이 가짜 정상을 학습하면 탐지 감도가 무뎌집니다.
- 긴 블록은 채우지 말고 **구간째 제외**하고, 윈도우 생성 시 결측에 걸친 윈도우를 버립니다.

</details>

---

### Q3. 홀드값(센서 고착)이 NaN보다 위험한 이유는 무엇입니까?

<details>
<summary>정답 보기</summary>

- NaN은 도구가 자동으로 세어 주지만, 홀드값은 **정상값처럼 생긴 죽은 데이터**이기 때문입니다.
- 통신은 살아 있으므로 수집 시스템 관점에서는 아무 경고가 없습니다.
- 동일값 연속 길이 검사(`detect_hold`)와 물리 범위 검증을 검진 루틴에 넣어야 잡을 수 있습니다.

</details>

---

### Q4. [현장 판단] 10초 주기 진동 센서를 10분 주기로 리샘플링해 대시보드에 쓰려 합니다. 집계 함수는 무엇으로 골라야 합니까?

<details>
<summary>정답 보기</summary>

- 목적에 따라 다릅니다 — 평균 부하 추이가 목적이면 mean, **이상 징후 감시가 목적이면 max**입니다.
- 순간 스파이크(충돌·베어링 손상 징후)는 mean 집계에서 사라집니다.
- 상태 누적값(공구 사용 시간 등)은 last가 자연스럽습니다. 컬럼별로 다르게 고르는 것이 정답에 가깝습니다.

</details>


---
## 다음 노트북 예고 — NB02. 딥러닝 시계열 계보

토대가 끝났으니 이제 모델입니다. 다음 노트북의 질문은 하나입니다.

> **"Transformer는 갑자기 나타나지 않았다. 순환을 버려야 했던 이유가 있었다."**

RNN이 왜 한계에 부딪혔는지, Attention이 무엇을 바꿨는지,
그리고 시계열 전용 Transformer인 **PatchTST**가 왜 "패치"로 써는지 —
오늘 만든 `make_windows()` 가 그 이야기의 출발점이 됩니다.
항공기 엔진의 잔여수명(RUL) 예측이 무대입니다.